In [46]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


In [3]:
np.random.seed(42)
tf.random.set_seed(42)

In [25]:
train_data_dir = 'C:\\Users\\HP\\Downloads\\archive (3)\\Industrial-Equipment\\Defected'
test_data_dir = 'C:\\Users\\HP\\Downloads\\archive (3)\\Industrial-Equipment\\Non-Defected'

In [52]:
def load_images_from_directory(directory):
    images = []
    labels = []
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if os.path.isdir(label_dir):
            print("Loading images from label:", label)
            for filename in os.listdir(label_dir):
                img_path = os.path.join(label_dir, filename)
                print("Loading image:", img_path)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.resize(img, (150, 150))  # Resize images if necessary
                    images.append(img)
                    labels.append(int(label))  # Assuming directory names represent labels (0 for non-defected, 1 for defected)
                else:
                    print("Failed to load image:", img_path)
    return np.array(images), np.array(labels)


In [53]:
x_train, y_train = load_images_from_directory(train_data_dir)
x_test, y_test = load_images_from_directory(test_data_dir)

In [54]:
train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
test_datagen = ImageDataGenerator(rescale=1./255)

In [55]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [56]:
print("Shape of training input:", x_train.shape)
print("Shape of training output:", y_train.shape)
print("Shape of testing input:", x_test.shape)
print("Shape of testing output:", y_test.shape)

Shape of training input: (0,)
Shape of training output: (0,)
Shape of testing input: (0,)
Shape of testing output: (0,)


In [27]:
img_width, img_height = 150, 150
batch_size = 32

In [28]:
train_generator = train_datagen.flow_from_directory(train_data_dir, target_size=(img_width, img_height),batch_size=batch_size, class_mode='binary')

Found 0 images belonging to 0 classes.


In [29]:
test_generator = test_datagen.flow_from_directory(test_data_dir, target_size=(img_width, img_height),batch_size=batch_size, class_mode='binary')

Found 0 images belonging to 0 classes.


In [30]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),  
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),  
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])


In [31]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [32]:
print("Number of samples in train generator:", len(train_generator))
print("Number of samples in test generator:", len(test_generator))


Number of samples in train generator: 0
Number of samples in test generator: 0


In [33]:
x_batch, y_batch = next(train_generator)
print("Shape of input batch:", x_batch.shape)
print("Shape of output batch:", y_batch.shape)


Shape of input batch: (0, 150, 150, 3)
Shape of output batch: (0,)


In [42]:
steps_per_epoch = len(train_generator)
validation_steps = len(test_generator)

In [ ]:
history = model.fit(train_generator, epochs=10, steps_per_epoch=steps_per_epoch,
                    validation_data=test_generator, validation_steps=validation_steps)

In [39]:
x_batch_train, y_batch_train = next(train_generator)
x_batch_test, y_batch_test = next(test_generator)

print("Shape of training input batch:", x_batch_train.shape)
print("Shape of training output batch:", y_batch_train.shape)
print("Shape of testing input batch:", x_batch_test.shape)
print("Shape of testing output batch:", y_batch_test.shape)


Shape of training input batch: (0, 150, 150, 3)
Shape of training output batch: (0,)
Shape of testing input batch: (0, 150, 150, 3)
Shape of testing output batch: (0,)


In [ ]:
score = model.evaluate(test_generator)
print("Test Loss:", score[0])
print("Test Accuracy:", score[1])